# 1 · Data

Download the datasets and unpack them into the layout `original_code/load_data/` expects. This is the desktop counterpart of `colab_datasets.ipynb`, and the download reuses `download_datasets.py`.

Each dataset section runs on its own, so you can start with Caltech-256 alone. Every step skips work that is already done. Start Jupyter or VS Code from the repository root.

## Environment (once)

Use Python 3.12; the release pins `numpy==1.26.4`, which has no wheels for newer versions. PyPI's Windows builds of torch are CPU-only, so install torch from the CUDA index first, then the release's requirements. In PowerShell, from the repository root:

```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126
pip install -r original_code/requirements.txt
pip install notebook
```

Then choose `.venv` as this notebook's kernel.

In [ ]:
import os
import shutil
import sys
import tarfile
from pathlib import Path

REPO = Path.cwd()
assert (REPO / "original_code").is_dir(), "Start Jupyter from the repository root."
for path in (REPO, REPO / "original_code"):  # the release imports its packages by top-level name
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# Where the datasets go (about 15 GB): set DD2610_DATA, or edit the default.
DATA_ROOT = Path(os.environ.get("DD2610_DATA", Path.home() / "dd2610-data")).resolve()
ARCHIVES = DATA_ROOT / "datasets"  # download_datasets.py saves the archives here
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(DATA_ROOT)

## Download

This uses the same helper as the Colab notebook, and files already present are skipped. Caltech-256 (1.2 GB), ImageNetV2 matched-frequency (1.2 GB) and the ImageNet devkit come from public URLs, with checksums.

ImageNet-1K validation (6.4 GB) needs the team-restricted URL in the `IMAGENET_VAL_URL` environment variable. Alternatively, copy `ILSVRC2012_img_val.tar` into the `datasets` folder by hand.

In [ ]:
from download_datasets import download_datasets

download_datasets(DATA_ROOT, os.environ.get("IMAGENET_VAL_URL"))

In [ ]:
def unpack(archive: Path, target: Path) -> None:
    """Extract an archive into target. Extraction goes to a temporary folder first,
    so an interrupted run never leaves a half-filled target behind."""
    if target.exists():
        print(f"{target} exists, skipping")
        return
    partial = target.with_name(target.name + ".partial")
    shutil.rmtree(partial, ignore_errors=True)
    print(f"Extracting {archive.name} ...", flush=True)
    with tarfile.open(archive) as tar:
        tar.extractall(partial, filter="data")
    partial.rename(target)

## Caltech-256

The release reads Caltech-256 through torchvision's `Caltech256(root=<data root>)`, which expects `<data root>/caltech256/256_ObjectCategories/`. The check builds the release's own 70/15/15 split (seed 42): 21,424 training, 4,591 validation and 4,592 test images.

In [ ]:
unpack(ARCHIVES / "256_ObjectCategories.tar", DATA_ROOT / "caltech256")

from load_data.datasets import Caltech256Dataset

# The release's loaders need num_workers > 0. Only the split sizes are read here, so no worker starts.
caltech = Caltech256Dataset(root_dir=str(DATA_ROOT)).get_splits(transforms=None, batch_size=128, num_workers=1)
sizes = {split: len(loader.dataset) for split, loader in caltech.items() if loader is not None}
print(sizes, "total", sum(sizes.values()))
assert sum(sizes.values()) == 30607, "expected 30,607 images"
image, label = caltech["train"].dataset[0]
print("first training image:", image.size, image.mode, "label", label)

## ImageNetV2 (matched frequency)

ImageNetV2 is the distribution-shift test set for the ImageNet models. The release's `IntegerLabelDataset` reads it from class folders named 0–999.

In [ ]:
unpack(ARCHIVES / "imagenetv2-matched-frequency.tar.gz", DATA_ROOT / "imagenetv2")
IMAGENET_V2 = DATA_ROOT / "imagenetv2" / "imagenetv2-matched-frequency-format-val"

from load_data.datasets import IntegerLabelDataset

v2 = IntegerLabelDataset(IMAGENET_V2)
labels = {int(path.parent.name) for path in v2.fnames}
print(len(v2), "images,", len(labels), "classes")
assert len(v2) == 10000 and labels == set(range(1000))

## ImageNet-1K validation

This section is skipped until `ILSVRC2012_img_val.tar` is in the `datasets` folder. torchvision's own helpers check the archives' checksums and sort the 50,000 images into class folders (`val/<wnid>/`) using the devkit. Sorted wnids give the standard class indices.

The release's `ImageNetDataset` builds an `ImageFolder` on `train/` even when it only evaluates. A placeholder `train/` holding one copied validation image is therefore created; nothing reads it. The check builds the release's 5%/95% validation/test split (seed 42) together with its ImageNetV2 loader.

In [ ]:
from torchvision.datasets.imagenet import load_meta_file, parse_devkit_archive, parse_val_archive

IMAGENET = DATA_ROOT / "imagenet"
val_dir = IMAGENET / "val"
if not (ARCHIVES / "ILSVRC2012_img_val.tar").is_file():
    print("ILSVRC2012_img_val.tar not found; skipping ImageNet-1K.")
else:
    if not val_dir.exists():
        if not (ARCHIVES / "meta.bin").is_file():
            parse_devkit_archive(ARCHIVES)  # checks the devkit's md5 and writes meta.bin
        wnids = load_meta_file(ARCHIVES)[1]
        partial = IMAGENET / "val.partial"
        shutil.rmtree(partial, ignore_errors=True)
        print("Checking and extracting ILSVRC2012_img_val.tar ...", flush=True)
        parse_val_archive(ARCHIVES, wnids=wnids, folder=str(partial))  # checks the archive's md5
        partial.rename(val_dir)
    placeholder = IMAGENET / "train"
    if not placeholder.exists():
        first_class = min(val_dir.iterdir())
        (placeholder / first_class.name).mkdir(parents=True)
        shutil.copy2(min(first_class.iterdir()), placeholder / first_class.name)

    from load_data.datasets import ImageNetDataset

    imagenet = ImageNetDataset(root_dir=str(IMAGENET), ood_root_dir=str(IMAGENET_V2)).get_splits(
        transforms=None, batch_size=32, num_workers=1)
    sizes = {split: len(loader.dataset) for split, loader in imagenet.items() if split != "train"}
    print(sizes)
    assert sizes == {"val": 2500, "test": 47500, "ood_test": 10000}

## Summary

In [ ]:
for name, path in [("Caltech-256", DATA_ROOT / "caltech256" / "256_ObjectCategories"),
                   ("ImageNetV2", DATA_ROOT / "imagenetv2" / "imagenetv2-matched-frequency-format-val"),
                   ("ImageNet-1K val", DATA_ROOT / "imagenet" / "val")]:
    print(f"{name:16s} {'ready' if path.is_dir() else 'missing':8s} {path}")